# Decorators

<div style="display:flex;justify-content:space-between;align-items:center;background:#0d0d0f;border:1px solid #1e1e24;border-left:4px solid #4fc3f7;border-radius:6px;padding:clamp(1rem,2.5vw,1.8rem) clamp(1.2rem,3vw,2.4rem);margin-bottom:2rem;position:relative;overflow:hidden;box-shadow:0 4px 32px rgba(0,0,0,0.5);font-family:'Segoe UI',sans-serif;">
<div style="position:absolute;top:0;left:0;right:0;bottom:0;background:radial-gradient(ellipse at 0% 50%,rgba(79,195,247,0.07) 0%,transparent 60%);pointer-events:none;"></div>
<div style="display:flex;flex-direction:column;gap:0.3rem;">
<p style="font-size:clamp(1.3rem,3.5vw,2.4rem);color:#f0f0f5;margin:0;line-height:1.1;font-weight:700;letter-spacing:-0.01em;">Decorators</p>
<p style="font-size:clamp(0.75rem,1.6vw,1rem);color:#7a7a90;margin:0;letter-spacing:0.04em;font-weight:300;">Development Expert Python / PCAP &nbsp;|&nbsp; Kapitel 3: Funktionen &nbsp;|&nbsp; Notebook 03d</p>
</div>
</div>

**Legende**

> **[Kursinhalt]** Dieses Notebook ist kein PCAP-Pruefungsinhalt

---

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
1. Das Problem: Code um Funktionen herum
</span>
</div>

Stell dir vor, du hast zehn Funktionen in deinem Projekt und moechtest fuer jede messen, wie lange sie braucht. Ohne Decorators wuerdest du so etwas tun:

In [ ]:
import time

def berechne_summe(n):
    start = time.time()       # Zeitmessung: Anfang
    ergebnis = sum(range(n))
    ende = time.time()        # Zeitmessung: Ende
    print(f'Laufzeit: {ende - start:.4f}s')
    return ergebnis

def berechne_produkt(zahlen):
    start = time.time()       # Dieselbe Zeitmessung nochmal...
    ergebnis = 1
    for z in zahlen:
        ergebnis *= z
    ende = time.time()        # ...und nochmal
    print(f'Laufzeit: {ende - start:.4f}s')
    return ergebnis

berechne_summe(1_000_000)
berechne_produkt(range(1, 100))

Das Zeitmessung-Boilerplate wiederholt sich in jeder Funktion. Das verletzt ein wichtiges Prinzip: **DRY -- Don't Repeat Yourself**.

Und wenn wir spaeter die Zeitmessung aendern wollen (z.B. in eine Log-Datei schreiben statt `print`), muessen wir jede Funktion anfassen.

Es muss einen besseren Weg geben.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
2. Die Loesung: Funktionen in Funktionen einwickeln
</span>
</div>

Was wenn wir die Zeitmessung auslagern -- in eine Funktion, die eine andere Funktion entgegennimmt, sie mit Zeitmessung umhuellt und die neue Version zurueckgibt?

Wir haben alle Werkzeuge dafuer bereits: hoeherwertige Funktionen (03b) und Closures (03c).

In [ ]:
import time

# Die Wrapper-Funktion nimmt eine Funktion entgegen
# und gibt eine neue, erweiterte Version zurueck
def mit_zeitmessung(funktion):
    def wrapper(*args, **kwargs):         # *args und **kwargs: alle Argumente weiterleiten
        start = time.time()
        ergebnis = funktion(*args, **kwargs)  # die Originalfunktion aufrufen
        ende = time.time()
        print(f'{funktion.__name__} lief {ende - start:.4f}s')
        return ergebnis
    return wrapper

# Funktionen ohne Boilerplate
def berechne_summe(n):
    return sum(range(n))

def berechne_produkt(zahlen):
    ergebnis = 1
    for z in zahlen:
        ergebnis *= z
    return ergebnis

# Jetzt die Zeitmessung drum herumwickeln
berechne_summe    = mit_zeitmessung(berechne_summe)
berechne_produkt  = mit_zeitmessung(berechne_produkt)

berechne_summe(1_000_000)
berechne_produkt(range(1, 100))

Das funktioniert. Die Originalfunktionen enthalten kein Zeitmessungs-Boilerplate mehr. 

Aber diese Zeile ist etwas unschoen:
```python
berechne_summe = mit_zeitmessung(berechne_summe)
```

Das Muster -- *definiere Funktion, wickle sie sofort ein und weise sie wieder demselben Namen zu* -- ist so haeufig, dass Python dafuer eine eigene Syntax erfunden hat.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
3. Die @-Syntax
</span>
</div>

Das `@`-Symbol vor einer Funktionsdefinition ist **syntaktischer Zucker** -- eine kuerzere Schreibweise fuer exakt dasselbe Muster von oben.

```python
@mit_zeitmessung
def berechne_summe(n):
    return sum(range(n))
```

ist **identisch** mit:

```python
def berechne_summe(n):
    return sum(range(n))
berechne_summe = mit_zeitmessung(berechne_summe)
```

Python sieht `@mit_zeitmessung`, definiert die Funktion darunter, ruft `mit_zeitmessung(berechne_summe)` auf und haengt das Ergebnis an den Namen `berechne_summe`.

In [ ]:
import time

def mit_zeitmessung(funktion):
    def wrapper(*args, **kwargs):
        start = time.time()
        ergebnis = funktion(*args, **kwargs)
        ende = time.time()
        print(f'{funktion.__name__} lief {ende - start:.4f}s')
        return ergebnis
    return wrapper

# Mit @-Syntax: sauber und klar
@mit_zeitmessung
def berechne_summe(n):
    return sum(range(n))

@mit_zeitmessung
def berechne_produkt(zahlen):
    ergebnis = 1
    for z in zahlen: ergebnis *= z
    return ergebnis

berechne_summe(1_000_000)
berechne_produkt(range(1, 100))

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
4. Das Problem mit dem Namen -- functools.wraps
</span>
</div>

Es gibt ein subtiles Problem mit unserem Decorator. Nach dem Dekorieren zeigt `berechne_summe` eigentlich auf `wrapper` -- die innere Funktion. Damit geht der Name der Originalfunktion verloren:

In [ ]:
print(berechne_summe.__name__)  # 'wrapper' -- nicht 'berechne_summe'!
print(berechne_summe.__doc__)   # None -- Docstring auch weg

Das ist ein Problem sobald man `__name__` oder den Docstring braucht -- z.B. in Fehlermeldungen, Logs oder bei der automatischen Dokumentation.

`functools.wraps` loest das: Es kopiert die Metadaten der Originalfunktion auf den Wrapper.

In [ ]:
import time
from functools import wraps

def mit_zeitmessung(funktion):
    @wraps(funktion)  # kopiert __name__, __doc__ usw. auf wrapper
    def wrapper(*args, **kwargs):
        start = time.time()
        ergebnis = funktion(*args, **kwargs)
        ende = time.time()
        print(f'{funktion.__name__} lief {ende - start:.4f}s')
        return ergebnis
    return wrapper

@mit_zeitmessung
def berechne_summe(n):
    """Berechnet die Summe aller Zahlen von 0 bis n."""
    return sum(range(n))

print(berechne_summe.__name__)  # 'berechne_summe' -- jetzt korrekt
print(berechne_summe.__doc__)   # 'Berechnet die Summe...' -- Docstring bleibt erhalten
berechne_summe(1_000_000)

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
5. Weitere Praxisbeispiele
</span>
</div>

In [ ]:
from functools import wraps

# Decorator: Eingaben validieren
def nur_positive_zahlen(funktion):
    @wraps(funktion)
    def wrapper(*args, **kwargs):
        for arg in args:
            if isinstance(arg, (int, float)) and arg <= 0:
                raise ValueError(f'Nur positive Zahlen erlaubt, bekommen: {arg}')
        return funktion(*args, **kwargs)
    return wrapper

@nur_positive_zahlen
def wurzel(x):
    import math
    return math.sqrt(x)

@nur_positive_zahlen
def logarithmus(x):
    import math
    return math.log(x)

print(wurzel(16))       # 4.0
print(logarithmus(10))  # 2.302...

try:
    wurzel(-4)
except ValueError as e:
    print(f'Fehler: {e}')

In [ ]:
from functools import wraps

# Decorator mit eigenem Parameter -- eine Ebene tiefer verschachtelt
def wiederhole(n_mal):
    """Decorator-Factory: gibt einen Decorator zurueck der n_mal wiederholt."""
    def decorator(funktion):
        @wraps(funktion)
        def wrapper(*args, **kwargs):
            letztes_ergebnis = None
            for _ in range(n_mal):
                letztes_ergebnis = funktion(*args, **kwargs)
            return letztes_ergebnis
        return wrapper
    return decorator  # gibt den eigentlichen Decorator zurueck

@wiederhole(3)
def begruessung(name):
    print(f'Hallo, {name}!')

begruessung('Alice')

Der Decorator mit Parametern hat eine Ebene mehr: `wiederhole(3)` ist keine Decorator-Funktion selbst -- es ist eine **Decorator-Factory** die einen Decorator erzeugt. `@wiederhole(3)` ruft zuerst `wiederhole(3)` auf, bekommt `decorator` zurueck, und wendet diesen dann auf `begruessung` an.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
6. Decorators in der Praxis -- wo begegnen sie uns?
</span>
</div>

Decorators sind ueberall in Python -- auch wenn man sie nicht immer als solche erkennt:

```python
class MeineKlasse:
    @staticmethod          # eingebauter Decorator
    def hilfsmethode():
        pass

    @classmethod           # eingebauter Decorator
    def fabrik(cls):
        pass

    @property              # eingebauter Decorator
    def wert(self):
        return self._wert
```

```python
from functools import lru_cache

@lru_cache(maxsize=128)    # automatisches Caching von Ergebnissen
def fibonacci(n):
    if n < 2: return n
    return fibonacci(n-1) + fibonacci(n-2)
```

In Web-Frameworks wie Flask:
```python
@app.route('/startseite')  # URL-Handler registrieren
def startseite():
    return 'Willkommen!'
```

Das Muster dahinter ist immer dasselbe: Eine Funktion wird entgegengenommen, erweitert und zurueckgegeben -- ohne den Originalcode zu veraendern.

---

**Zusammenfassung**

| Konzept | Erklaerung |
|---------|------------|
| Decorator | Funktion die eine andere Funktion einwickelt und erweitert |
| `@name` | Syntaktischer Zucker fuer `funktion = name(funktion)` |
| `*args, **kwargs` | Alle Argumente transparent weiterleiten |
| `@wraps(fn)` | Metadaten (`__name__`, `__doc__`) der Originalfunktion erhalten |
| Decorator-Factory | Funktion die einen Decorator erzeugt -- benoetigt wenn der Decorator Parameter hat |

**Was dahintersteckt:** Ein Decorator kombiniert alles aus diesem Kapitel -- Funktionen als Objekte (03a), hoeherwertige Funktionen (03b) und Closures (03c). Das `@`-Symbol ist nur ein bequemer Shortcut fuer ein Muster, das wir selbst haetten erfinden koennen.

---

**Weiter:** In Notebook **03e -- Lambda, map und filter** lernen wir kompakte anonyme Funktionen und die klassischen funktionalen Werkzeuge von Python kennen.